# A minor change to avoid data leakgae in PCA
### We shouldnt calculate PCA once using all subjects before K-fold. Instead, calculate PCA separately inside each training fold.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNet
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ============================================================
# 1. Load dataset
# ============================================================
df = pd.read_csv("wide_diff_all_data.csv")
print("Number of subjects:", len(df))

# ============================================================
# 2. Define diffusion features
# ============================================================
diffusion_substrings = [
    "dki_fa",
    "dki_md",
    "dki_mk",
    "dki_awf"
]

diffusion_cols = [
    col for col in df.columns
    if any(x in col for x in diffusion_substrings)
]

print("Number of diffusion features:", len(diffusion_cols))

# ============================================================
# 3. Prepare outcome and covariates
# ============================================================

# Outcome
y = pd.to_numeric(
    df["DDisc_AUC_40K"],
    errors="coerce"
)


# Covariates
covariates = df[
    [
        "age_group",
        "Gender"
    ]
].copy()


# Convert categorical variables
covariates = pd.get_dummies(
    covariates,
    columns=["age_group", "Gender"],
    drop_first=True
)


# ============================================================
# 4. Create feature matrix
# ============================================================
X_diff = df[diffusion_cols].copy()

# make sure diffusion variables are numeric
X_diff = X_diff.apply(
    pd.to_numeric,
    errors="coerce"
)


# combine MRI + age + sex
X = pd.concat(
    [
        X_diff,
        covariates
    ],
    axis=1
)


# remove subjects with missing values
complete = pd.concat(
    [
        X,
        y
    ],
    axis=1
).dropna()


X = complete.drop(
    columns=["DDisc_AUC_40K"]
)

y = complete["DDisc_AUC_40K"].astype(float)

print("Final subjects:", len(y))
print("Final features:", X.shape[1])

# ============================================================
# 5. Define nested cross validation
# ============================================================

# Outer CV:
# estimates final model performance
outer_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# Inner CV:
# chooses best hyperparameters
inner_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# ============================================================
# 6. Create leakage-free pipeline
# ============================================================
pipeline = Pipeline(
    [
        # scaling happens only using training subjects
        (
            "scaler",
            StandardScaler()
        ),

        # PCA is fitted only inside each training fold
        (
            "pca",
            PCA()
        ),

        # prediction model
        (
            "model",
            ElasticNet(
                max_iter=10000
            )
        )
    ]
)


# ============================================================
# 7. Hyperparameter search
# ============================================================

param_grid = {
    # number of PCA components
    "pca__n_components":
        [
            5,
            10,
            15,
            20,
            25
        ],
    # ElasticNet regularization
    "model__alpha":
        np.logspace(-3, 2, 20 ),
    # balance between Ridge and Lasso
    "model__l1_ratio":
        np.linspace(0.1, 0.9, 9 )
}

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=inner_cv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)


# ============================================================
# 8. Nested CV prediction
# ============================================================
y_pred = cross_val_predict( search, X,    y,
    cv=outer_cv,
    n_jobs=-1
)


# ============================================================
# 9. Performance
# ============================================================
rmse = np.sqrt(mean_squared_error(y,y_pred))
mae = mean_absolute_error(y,y_pred)
r2 = r2_score(y,y_pred)


print("==============================")
print("Nested CV Results")
print("==============================")

print(f"R²   : {r2:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
df["DDisc_AUC_40K"].describe()
# ============================================================
# 10. Train final model on all data
#     (optional, after evaluation)
# ============================================================
search.fit(X,y)

print("\nBest parameters:")
print(search.best_params_)

Number of subjects: 691
Number of diffusion features: 96
Final subjects: 691
Final features: 100
Nested CV Results
R²   : -0.0139
RMSE : 0.2855
MAE  : 0.2462

Best parameters:
{'model__alpha': 0.23357214690901212, 'model__l1_ratio': 0.7000000000000001, 'pca__n_components': 5}
